# Blackbox-LSTM GR predictor on **LA2A** (SignalTrain)

**Google Colab**: Runtime → **GPU**. Open via *File → Open notebook → GitHub*
(`5aola/Virtual-Analogue-Compressor-Modelling`); cell 1 clones the repo and mounts
Drive for the dataset. **Push local changes before running.**

## What this is

A **retarget of [`05_conditioning/train_lstm_blackbox_gr.ipynb`](../05_conditioning/train_lstm_blackbox_gr.ipynb)**
— the learned-frontend frame-rate **gain-reduction predictor** — from Diff-SSL-G-Comp
to the **Teletronix LA-2A** (SignalTrain 1.1 dataset). The model, the loss/system,
and the crop/train regime are the *same* `05_conditioning` code, unchanged. Only the
dataset, its split, and the conditioning width (2 LA2A knobs instead of 4) differ.

```
dry ─ learned Conv frame-encoder (stride 256) ─ dilated temporal Conv stack ─ 1×1 proj
      features → LSTM(8→32) → polynomial-FiLM(knobs) → GLU → Linear → gr [dB]
```

The predictor emits GR at frame rate (172 Hz, hop 256); the `DetectorGRSystem` loss
is a masked `Huber(dB) + 1.0·Huber(ΔdB)` on the pooled target, with an energy floor
and a crop-start warmup mask. This is the front half of the LA2A cascade — its
predicted GR feeds the waveshaper gain-prior model
([`08_la2a/train_lstm_gain_prior_ws.ipynb`](train_lstm_gain_prior_ws.ipynb)).

## Dataset — SignalTrain LA2A (`data/LA2A/all/`)

84 **long recordings** (4-20 min each, 44.1 kHz mono float), one per
`(Comp/Limit, Peak Reduction)` setting: `input_<id>_.wav` (dry) +
`target_<id>_LA2A_<Nc>__<cl>__<pr>.wav` (wet). **2 knobs** (`la2a_info.ini`):
Comp/Limit ∈ {0,1} switch, Peak Reduction ∈ {0,5,..,100}. 42 unique settings.

**GR is recomputed on-the-fly** from each `(dry, wet)` crop with the *exact* export
function — `src.dsp_torch.gain_reduction_db(dry, wet, 1024)` (causal
zero-left-padded 1024-RMS; the function `03_initial_GR_pred` used to write the
`gr_curves/*.pt`). A 1023-sample lookback fills the causal window from real
preceding samples, so each crop's GR is **bit-identical to slicing the full-file
curve**. No `.pt` cache — only the WAVs are needed. The GR-predictor contract
`(dry, gr, params)` is produced by `08_la2a/dataset_la2a_gr.py`, which reuses the
gain-prior LA2A crop/GR machinery and just drops `wet`.

## Split — percentage-based, temporal within each recording (`08_la2a/splits_la2a.py`)

Diff-SSL's song-level / `test_ground_truth` policy does **not** transfer: every LA2A
setting lives in only one (a few in two-three) recording(s), so holding out whole
recordings would delete a setting from training and break the knob conditioning.
Instead we split **temporally by time fraction within each recording** — the standard
LA2A methodology (SignalTrain, Steinmetz TCN, Comparative-Study, Optical-DRC all test
on held-out *audio regions* at the same settings):

```
per recording:  [0, 0.8) → train    [0.8, 0.9) → val    [0.9, 1.0) → test
```

- **Every setting is in all three splits** → conditioning fully learnable; the test
  set probes generalisation to *unseen audio at known settings* (the LA-2A question).
- **No content leakage** — boundaries are by time fraction, identical for every
  recording. Within each region crops are evenly-spaced and capped at `crops_per_pair`.

Otherwise identical to the `05_conditioning` blackbox GR run (same `BlackboxGRLSTM`
frontend, same `DetectorGRSystem` loss, same crop/warmup regime, fresh state per
batch) — reused unchanged.

In [1]:
# ── 0. Dependencies ──────────────────────────────────────────────────
# No nablafx needed for the GR predictor (model_blackbox_gr imports only
# gr_target; system_detector_gr is standalone). Pin numpy first so lightning
# installs can't downgrade Colab's numpy 2.x and break torch; install
# lightning --no-deps so it can't clobber Colab's CUDA torch.
!pip install -q "numpy>=2.0,<2.6"
!pip install -q torchmetrics soundfile lightning-utilities packaging
!pip install -q --no-deps lightning

import numpy as np, torch
assert np.__version__.startswith("2."), f"numpy {np.__version__} — restart runtime, re-run cell 0"
print(f"numpy {np.__version__}, torch {torch.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 55.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 62.8 MB/s eta 0:00:00
numpy 2.0.2, torch 2.11.0+cu128


In [2]:
# ── 1. Mount Drive (dataset) + clone repo from GitHub (code) ─────────
# The repo is NOT synced to Drive (only data/ is). Code comes from GitHub —
# push local changes before (re)running this cell; re-running pulls updates.

import os
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_DATA_ROOT = "/content/drive/Othercomputers/MacBook Air/data/LA2A"
REPO_URL = "https://github.com/5aola/Virtual-Analogue-Compressor-Modelling.git"
REPO_ROOT = "/content/Virtual-Analogue-Compressor-Modelling"

if os.path.isdir(REPO_ROOT):
    !git -C "{REPO_ROOT}" fetch origin
    !git -C "{REPO_ROOT}" reset --hard origin/main
else:
    !git clone --depth 1 "{REPO_URL}" "{REPO_ROOT}"

DATA_ROOT = DRIVE_DATA_ROOT

# Module dirs: LA2A dataset/split (08_la2a) + the GR-predictor model/system it
# reuses unchanged (05_conditioning). Both go on sys.path, plus the repo root
# for `src`.
LA2A_DIR = os.path.join(REPO_ROOT, "08_la2a")
COND_DIR = os.path.join(REPO_ROOT, "05_conditioning")
assert os.path.isfile(os.path.join(LA2A_DIR, "dataset_la2a_gr.py")), (
    f"Clone failed or stale: {LA2A_DIR}. Did you push local changes?"
)
assert os.path.isfile(os.path.join(COND_DIR, "model_blackbox_gr.py")), (
    f"Missing 05_conditioning model modules: {COND_DIR}"
)

OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), "la2a_gr_pred_runs")

assert os.path.isdir(os.path.join(DATA_ROOT, "all")), (
    f"No all/ under {DATA_ROOT} — sync the SignalTrain LA2A WAVs to Drive first."
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Drop cached local modules so a prior run cannot keep stale classes.
for _name in list(sys.modules):
    if _name in ("dataset_la2a", "dataset_la2a_gr", "splits_la2a",
                 "model_blackbox_gr", "system_detector_gr", "gr_target"):
        del sys.modules[_name]

for p in (REPO_ROOT, COND_DIR, LA2A_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"LA2A_DIR   : {LA2A_DIR}")
print(f"COND_DIR   : {COND_DIR}")
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

Mounted at /content/drive
Cloning into '/content/Virtual-Analogue-Compressor-Modelling'...
remote: Enumerating objects: 252, done.
remote: Counting objects: 100% (252/252), done.
remote: Compressing objects: 100% (235/235), done.
remote: Total 252 (delta 20), reused 163 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (252/252), 130.50 MiB | 15.53 MiB/s, done.
Resolving deltas: 100% (20/20), done.
Updating files: 100% (224/224), done.
REPO_ROOT  : /content/Virtual-Analogue-Compressor-Modelling
LA2A_DIR   : /content/Virtual-Analogue-Compressor-Modelling/08_la2a
COND_DIR   : /content/Virtual-Analogue-Compressor-Modelling/05_conditioning
DATA_ROOT  : /content/drive/Othercomputers/MacBook Air/data/LA2A
OUTPUT_DIR : /content/drive/Othercomputers/MacBook Air/data/la2a_gr_pred_runs


: 

In [3]:
# ── 2. Cache dataset to Colab local SSD ──────────────────────────────
# Only the input/target WAVs are cached (~29 GB float32). The 18 GB gr_curves/
# tree is deliberately NOT needed: dataset_la2a_gr recomputes GR on-the-fly per
# crop, bit-identical to the exported .pt. One-time copy per session.

import shutil
from dataset_la2a_gr import discover_la2a_pairs

LOCAL_DATA_ROOT = "/content/LA2A"
pairs = discover_la2a_pairs(DATA_ROOT)
print(f"Caching {len(pairs)} recordings (input+target WAVs) → {LOCAL_DATA_ROOT}")

def _mirror(src, dst):
    src, dst = Path(src), Path(dst)
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

for i, p in enumerate(pairs, 1):
    for key in ("dry", "wet"):
        _mirror(p[key], Path(LOCAL_DATA_ROOT) / Path(p[key]).relative_to(DATA_ROOT))
    if i % 10 == 0 or i == len(pairs):
        print(f"  cached {i}/{len(pairs)}")

DATA_ROOT = LOCAL_DATA_ROOT
print(f"Using local cache: {DATA_ROOT}")

Caching 84 recordings (input+target WAVs) → /content/LA2A
  cached 10/84
  cached 20/84
  cached 30/84
  cached 40/84
  cached 50/84
  cached 60/84
  cached 70/84
  cached 80/84
  cached 84/84
Using local cache: /content/LA2A


: 

In [4]:
# ── 3. Imports & hyper-parameters ────────────────────────────────────

import importlib
import json
from datetime import datetime

import torch
import torch.nn.functional as F
import lightning as pl
from lightning.pytorch.callbacks import (
    LearningRateMonitor, ModelCheckpoint, TQDMProgressBar,
)
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

import dataset_la2a as _dataset_la2a
importlib.reload(_dataset_la2a)
import dataset_la2a_gr as _dataset_la2a_gr
importlib.reload(_dataset_la2a_gr)
from dataset_la2a_gr import (
    BATCH_SIZE, RMS_WINDOW, SAMPLE_RATE, La2aGRCropDataModule, discover_la2a_pairs,
)

import splits_la2a as _splits_la2a
importlib.reload(_splits_la2a)
from splits_la2a import (
    LA2A_PARAM_ORDER, LA2A_PARAM_RANGES, build_la2a_split_manifest,
)

from model_blackbox_gr import BlackboxGRLSTM
from system_detector_gr import DetectorGRSystem
from gr_target import GR_DB_MAX, GR_DB_MIN

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "WARNING: CPU runtime")

# -- split: percentage-based, temporal within each recording --
SPLIT_SEED  = 42
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.8, 0.1, 0.1
# evenly-spaced crops per recording per split; caps the 24 h corpus to a
# trainable epoch. Raise for more data / more epochs of compute.
CROPS_PER_PAIR = {"train": 24, "val": 4, "test": 8}

# -- training: 5 s crops / 2 s warmup (LA-2A optical release is slow; a long
# warmup keeps frames after the mask from being supervised against a still-
# unknowable release tail). Fresh state per batch, fixed cosine budget. --
CROP_SEC = 5.0
SAMPLE_LENGTH = int(CROP_SEC * SAMPLE_RATE)
LR = 1e-3
MAX_EPOCHS = 200
SCHEDULER = "cosine"
ETA_MIN = 1e-6

HOP_SIZE = 256
WARMUP_SEC = 2.0
WARMUP_FRAMES = int(WARMUP_SEC * SAMPLE_RATE / HOP_SIZE)

# -- loss --
ENERGY_FLOOR_DB = -60.0   # frames with dry RMS below this are masked (noise labels)
HUBER_BETA_DB = 1.0       # quadratic below 1 dB error, linear above
DELTA_WEIGHT = 1.0        # first-difference term (attack/release timing)

# -- model (blackbox learned frontend; downstream == detector recipe) --
HIDDEN_SIZE = 32
NUM_CONTROLS = 2                        # LA2A: [comp_limit, peak_reduction]
ENC_CHANNELS = 16                       # learned frame-encoder width
FRONTEND_CHANNELS = 8                   # → LSTM(8→32)
TEMPORAL_KERNEL = 4
TEMPORAL_DILATIONS = (1, 4, 16, 64)     # causal temporal stack, RF 256 frames ≈ 1.49 s
FILM_ORDER = 3                          # polynomial FiLM

print(f"Crop {SAMPLE_LENGTH} ({SAMPLE_LENGTH/SAMPLE_RATE:.2f}s) | hop {HOP_SIZE} "
      f"({SAMPLE_RATE/HOP_SIZE:.0f} Hz frames) | warmup {WARMUP_FRAMES} frames "
      f"({WARMUP_SEC:.1f}s)")

RUN_TAG = "la2a_lstm_blackbox_gr_film"
RESUME_RUN = None

NVIDIA L4
Crop 220500 (5.00s) | hop 256 (172 Hz frames) | warmup 344 frames (2.0s)


: 

In [5]:
# ── 4. Preview split — percentage-based, temporal within each recording ─
# Every (comp_limit, peak_reduction) setting appears in train/val/test on
# DISJOINT temporal regions of its recording: conditioning is fully learnable
# and the test set is unseen audio at known settings (the LA2A convention).

pairs = discover_la2a_pairs(DATA_ROOT)
preview = build_la2a_split_manifest(
    pairs, seed=SPLIT_SEED, sample_length=SAMPLE_LENGTH,
    train_frac=TRAIN_FRAC, val_frac=VAL_FRAC, test_frac=TEST_FRAC,
    crops_per_pair=CROPS_PER_PAIR,
)

print(f"Recordings : {len(preview.pairs)}")
print(f"Settings   : {len(preview.settings)} unique [comp_limit, peak_reduction]")
print(f"  comp/limit values : {sorted({s[0] for s in preview.settings})}")
print(f"  peak_reduction    : {sorted({s[1] for s in preview.settings})}")
print(f"Regions    : train[0,{TRAIN_FRAC}) val[{TRAIN_FRAC},{round(TRAIN_FRAC+VAL_FRAC,3)}) "
      f"test[{round(TRAIN_FRAC+VAL_FRAC,3)},1) of every recording")
print(f"Crop totals: {preview.crop_counts}  (<= {CROPS_PER_PAIR} per recording)")
mins = {k: v * SAMPLE_LENGTH / SAMPLE_RATE / 60 for k, v in preview.crop_counts.items()}
print("Audio (min): " + "  ".join(f"{k}={mins[k]:.1f}" for k in ("train", "val", "test")))

assert all(preview.crop_counts[k] > 0 for k in ("train", "val", "test")), "empty split!"

Recordings : 84
Settings   : 42 unique [comp_limit, peak_reduction]
  comp/limit values : [0, 1]
  peak_reduction    : [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]
Regions    : train[0,0.8) val[0.8,0.9) test[0.9,1) of every recording
Crop totals: {'train': 2016, 'val': 336, 'test': 668}  (<= {'train': 24, 'val': 4, 'test': 8} per recording)
Audio (min): train=168.0  val=28.0  test=55.7


: 

In [6]:
# ── 5. Model size ────────────────────────────────────────────────────

model = BlackboxGRLSTM(
    hop_size=HOP_SIZE,
    sample_rate=SAMPLE_RATE,
    enc_channels=ENC_CHANNELS,
    frontend_channels=FRONTEND_CHANNELS,
    temporal_kernel=TEMPORAL_KERNEL,
    temporal_dilations=TEMPORAL_DILATIONS,
    hidden_size=HIDDEN_SIZE,
    num_controls=NUM_CONTROLS,
    film_order=FILM_ORDER,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"BlackboxGRLSTM: {n_params:,} params ({NUM_CONTROLS} LA2A knobs; "
      f"diffssl 4-knob run was 16,249)")
for name, mod in model.named_children():
    print(f"  {name:14s} {sum(p.numel() for p in mod.parameters()):,}")
_rf = model.temporal_context + 1
print(f"frontend: learned Conv encoder (stride {HOP_SIZE}) + dilated temporal stack "
      f"{list(TEMPORAL_DILATIONS)} — RF {_rf} frames (~{_rf * HOP_SIZE / SAMPLE_RATE:.2f} s)")
print(f"FiLM order: {FILM_ORDER}")

BlackboxGRLSTM: 16,121 params (2 LA2A knobs; diffssl 4-knob run was 16,249)
  frame_encoder  4,112
  temporal       4,160
  proj           136
  act            0
  lstm           5,376
  film           192
  glu            2,112
  head           33
frontend: learned Conv encoder (stride 256) + dilated temporal stack [1, 4, 16, 64] — RF 256 frames (~1.49 s)
FiLM order: 3


: 

In [7]:
# ── 6. DataModule + untrained-baseline check ─────────────────────────
# Reports the untrained model's masked GR MAE on a real val batch — the
# reference point training starts from (≈ mean-GR-magnitude, since the
# default-init head outputs ~0 dB).

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

assert DATA_ROOT.startswith("/content/"), "Run the cache cell first (cell 2)."

NUM_WORKERS = min(8, os.cpu_count() or 2)
print(f"DataLoader num_workers: {NUM_WORKERS}")

if RESUME_RUN:
    RUN_NAME = RESUME_RUN
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = os.path.join(RUN_DIR, "checkpoints", "last.ckpt")
    print(f"RESUMING: {RUN_NAME}")
else:
    RUN_NAME = f"la2a_gr_{datetime.now():%Y%m%d_%H%M%S}_{RUN_TAG}"
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = None
    print(f"NEW run: {RUN_NAME}")

os.makedirs(RUN_DIR, exist_ok=True)
split_path = os.path.join(RUN_DIR, "split_manifest.json")

dm = La2aGRCropDataModule(
    data_root=DATA_ROOT, sample_length=SAMPLE_LENGTH, sample_rate=SAMPLE_RATE,
    batch_size=BATCH_SIZE, split_seed=SPLIT_SEED,
    train_frac=TRAIN_FRAC, val_frac=VAL_FRAC, test_frac=TEST_FRAC,
    crops_per_pair=CROPS_PER_PAIR, rms_window=RMS_WINDOW,
    split_manifest_path=split_path, num_workers=NUM_WORKERS,
)
dm.setup()
print(f"Train/val/test crops: {len(dm.train_dataset)} / {len(dm.val_dataset)} / {len(dm.test_dataset)}")
print(f"Batches/epoch (train): {len(dm.train_dataloader())}  (batch_size={BATCH_SIZE})")

if not RESUME_RUN:
    _dry, _gr, _p = next(iter(dm.val_dataloader()))
    with torch.no_grad():
        _pred = model(_dry, _p)
    _n = _pred.shape[-1] * HOP_SIZE
    assert _pred.shape == (_dry.shape[0], 1, _dry.shape[-1] // HOP_SIZE), _pred.shape
    _tgt = F.avg_pool1d(_gr[..., :_n], HOP_SIZE)
    _en = 10 * torch.log10(F.avg_pool1d(_dry[..., :_n] ** 2, HOP_SIZE) + 1e-12) > ENERGY_FLOOR_DB
    _en[..., :WARMUP_FRAMES] = False
    print(f"untrained val-crop GR MAE: {float((_pred - _tgt).abs()[_en].mean()):.3f} dB"
          "  <- training starts here")
    del _dry, _gr, _p, _pred

DataLoader num_workers: 8
NEW run: la2a_gr_20260707_205937_la2a_lstm_blackbox_gr_film
Recordings     : 84
Settings       : 42 unique (comp_limit, peak_reduction)
Split fractions: train=0.8 val=0.1 test=0.1
Crops per rec  : {'train': 24, 'val': 4, 'test': 8}
Crop totals    : {'train': 2016, 'val': 336, 'test': 668}
La2aCropDataset[train]: 2016 crops from 84 recordings  [<= 24/rec, sample_length=220500, 168.0 min audio]
La2aCropDataset[val]: 336 crops from 84 recordings  [<= 4/rec, sample_length=220500, 28.0 min audio]
La2aCropDataset[test]: 668 crops from 84 recordings  [<= 8/rec, sample_length=220500, 55.7 min audio]
Train/val/test crops: 2016 / 336 / 668
Batches/epoch (train): 126  (batch_size=16)
untrained val-crop GR MAE: 1.390 dB  <- training starts here


: 

In [8]:
# ── 7. Train ─────────────────────────────────────────────────────────

with open(os.path.join(RUN_DIR, "hparams.json"), "w") as f:
    json.dump({
        "approach": "gr_prediction_learned_frontend",
        "model_type": "la2a_blackbox_gr_lstm_film",
        "model_ref": "05_conditioning blackbox GR predictor retargeted to LA2A",
        "dataset": "SignalTrain-LA2A",
        "setting": "multi (all 42 comp_limit x peak_reduction settings)",
        "conditioning": "film_glu_after_lstm (polynomial FiLM + GLU, Comparative-Study recipe)",
        "frontend": "learned_conv (frame-encoder stride 256 + dilated temporal stack)",
        "gr_source": "on-the-fly gain_reduction_db(dry, wet, 1024) == exported gr_curves/*.pt",
        "sample_rate": SAMPLE_RATE,
        "hop_size": HOP_SIZE,
        "rms_window": RMS_WINDOW,
        "sample_length": SAMPLE_LENGTH,
        "batch_size": BATCH_SIZE,
        "training": f"{CROP_SEC:g}s_crops_fresh_state_per_batch (warmup {WARMUP_SEC:g} s; no TBPTT)",
        "gr_range_db": [GR_DB_MIN, GR_DB_MAX],
        "param_order": LA2A_PARAM_ORDER,
        "param_ranges": LA2A_PARAM_RANGES,
        "split_seed": SPLIT_SEED,
        "split_policy": "temporal_within_recording (every setting in train/val/test; test = unseen audio regions)",
        "split_fracs": {"train": TRAIN_FRAC, "val": VAL_FRAC, "test": TEST_FRAC},
        "crops_per_pair": CROPS_PER_PAIR,
        "num_settings": len(dm.split.settings),
        "num_recordings": len(dm.split.pairs),
        "crop_counts": dm.split.crop_counts,
        "loss": {
            "kind": "masked_huber_delta",
            "energy_floor_db": ENERGY_FLOOR_DB,
            "huber_beta_db": HUBER_BETA_DB,
            "delta_weight": DELTA_WEIGHT,
            "warmup_frames": WARMUP_FRAMES,
        },
        "model": {
            "hidden_size": HIDDEN_SIZE,
            "num_controls": NUM_CONTROLS,
            "enc_channels": ENC_CHANNELS,
            "frontend_channels": FRONTEND_CHANNELS,
            "temporal_kernel": TEMPORAL_KERNEL,
            "temporal_dilations": list(TEMPORAL_DILATIONS),
            "film_order": FILM_ORDER,
            "num_params": n_params,
        },
        "lr": LR,
        "max_epochs": MAX_EPOCHS,
        "scheduler": SCHEDULER,
        "eta_min": ETA_MIN,
    }, f, indent=2)

system = DetectorGRSystem(
    model=model,
    lr=LR,
    warmup_frames=WARMUP_FRAMES,
    energy_floor_db=ENERGY_FLOOR_DB,
    huber_beta_db=HUBER_BETA_DB,
    delta_weight=DELTA_WEIGHT,
    scheduler=SCHEDULER,
    max_epochs=MAX_EPOCHS,
    eta_min=ETA_MIN,
)

ckpt_dir = os.path.join(RUN_DIR, "checkpoints")
callbacks = [
    ModelCheckpoint(
        dirpath=ckpt_dir, monitor="loss/val", mode="min", save_top_k=3,
        save_last=True, filename="best-{epoch:03d}-{step}",
        auto_insert_metric_name=False,
    ),
    LearningRateMonitor(logging_interval="epoch"),
    TQDMProgressBar(refresh_rate=10),
]
loggers = [
    TensorBoardLogger(save_dir=RUN_DIR, name="tb", version=""),
    CSVLogger(save_dir=RUN_DIR, name="csv", version=""),
]

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS, accelerator="gpu", devices=1,
    callbacks=callbacks, logger=loggers, gradient_clip_val=1.0,
    log_every_n_steps=10,
)

trainer.fit(system, dm, ckpt_path=_resume_ckpt)
print(f"Best val loss: {callbacks[0].best_model_score:.6f}")
print(f"Best ckpt    : {callbacks[0].best_model_path}")

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ BlackboxGRLSTM │ 16.1 K │ train │     0 │
└───┴───────┴────────────────┴────────┴───────┴───────┘

Trainable params: 16.1 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 16.1 K                                                                                               
Total estimated model params size (MB): 0.064                                                                      
Modules in train mode: 13                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

INFO: 
Detected KeyboardInterrupt, attempting graceful shutdown ...
INFO:lightning.pytorch.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/call.py", line 49, in _call_and_handle_interrupt
    return trainer_fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/trainer.py", line 630, in _fit_impl
    self._run(model, ckpt_path=ckpt_path, weights_only=weights_only)
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/trainer.py", line 1079, in _run
    results = self._run_stage()
              ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/trainer.py", line 1123, in _run_stage
    self.fit_loop.run()
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py", line 217, in run
    self.advance()
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py", line 469, in advance
    self.epoch_loop.run(self._data_fetcher)
  File

TypeError: object of type 'NoneType' has no len()

: 

In [ ]:
# ── 8. Test (held-out temporal regions — unseen audio at known settings) ─

trainer.test(system, datamodule=dm, ckpt_path=callbacks[0].best_model_path)

: 

In [ ]:
# ── 9. Full-region streaming eval + plot ─────────────────────────────
# Streams each recording's held-out region chunked with explicit state carry
# (the deployment path — chunked forward is exact vs one full forward). GR is
# recomputed on-the-fly per region (1023-sample lookback → bit-identical to the
# exported curve). A warmup mask drops the cold-start frames; MAE is over
# energy-valid, post-warmup frames — directly comparable to the run table.

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf

from src.dsp_torch import gain_reduction_db

best = torch.load(callbacks[0].best_model_path, map_location="cuda", weights_only=False)
system.load_state_dict(best["state_dict"])
system.eval().cuda()
model_e = system.model
print(f"Loaded best checkpoint: {callbacks[0].best_model_path}")

CHUNK = int(10.0 * SAMPLE_RATE) // HOP_SIZE * HOP_SIZE
LOOKBACK = RMS_WINDOW - 1


def _load_slice(path, start, stop):
    audio, _ = sf.read(path, start=start, stop=stop, dtype="float32", always_2d=True)
    x = torch.from_numpy(audio.T)
    if x.shape[0] > 1:
        x = x.mean(dim=0, keepdim=True)
    return x


@torch.no_grad()
def stream_region(m):
    start, end = m["region"]
    read_start = max(0, start - LOOKBACK)
    lb = start - read_start
    dry = _load_slice(m["dry"], read_start, end)   # [1, lb + L]
    wet = _load_slice(m["wet"], read_start, end)
    gr = gain_reduction_db(dry, wet, RMS_WINDOW)   # [1, lb + L]
    # drop the RMS lookback → region-aligned, then trim to whole frames
    dry, gr = dry[..., lb:], gr[..., lb:]
    n = min(dry.shape[-1], gr.shape[-1])
    n -= n % HOP_SIZE
    dry, gr = dry[None, ..., :n], gr[None, ..., :n]
    params = torch.tensor(m["params"], dtype=torch.float32)[None].cuda()

    state, preds = None, []
    for o in range(0, n, CHUNK):
        p, state = model_e(dry[..., o:o + CHUNK].cuda(), params, state, return_state=True)
        preds.append(p.cpu())
    pred = torch.cat(preds, -1)
    tgt = F.avg_pool1d(gr, HOP_SIZE)
    en = 10 * torch.log10(F.avg_pool1d(dry**2, HOP_SIZE) + 1e-12) > ENERGY_FLOOR_DB
    if WARMUP_FRAMES > 0:
        en = en.clone()
        en[..., :WARMUP_FRAMES] = False
    return pred, tgt, en


for split_name in ("val", "test"):
    errs, wsum = 0.0, 0
    for m in dm.meta[split_name]:
        pred, tgt, en = stream_region(m)
        if int(en.sum()) == 0:
            continue
        e = (pred - tgt).abs()[en]
        errs += float(e.sum())
        wsum += int(en.sum())
    print(f"{split_name.upper()} full-region GR MAE: {errs / max(wsum, 1):.3f} dB "
          f"({len(dm.meta[split_name])} recordings)")

# -- plot: mid-region window of one test recording, prediction vs target ------
pick = dm.meta["test"][len(dm.meta["test"]) // 2]
pred, tgt, en = stream_region(pick)
Tf = pred.shape[-1]
w0 = max(WARMUP_FRAMES, Tf // 2)
w1 = min(Tf, w0 + int(15.0 * SAMPLE_RATE / HOP_SIZE))  # 15 s window
t = np.arange(w0, w1) * HOP_SIZE / SAMPLE_RATE

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t, tgt[0, 0, w0:w1], label="Target GR", alpha=0.8, lw=0.7)
ax.plot(t, pred[0, 0, w0:w1], label="Predicted GR", alpha=0.8, lw=0.7)
m_win = en[..., w0:w1]
l1 = float((pred - tgt).abs()[..., w0:w1][m_win].mean()) if int(m_win.sum()) else float("nan")
ax.set_title(f"rec {pick['song']} / {pick['setting']} — window L1 = {l1:.2f} dB")
ax.set_xlabel("Time (s)")
ax.set_ylabel("GR (dB)")
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
plot_path = os.path.join(RUN_DIR, "eval_gr_comparison.png")
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Saved plot -> {plot_path}")
plt.show()

: 

In [ ]:
# ── 10. Frontend summary ─────────────────────────────────────────────
# Blackbox control: the frontend is fully learned (no fixed detector τ to
# inspect). Its causal receptive field sets how much level context the convs
# see before the LSTM; the LSTM supplies the rest of the memory.

_rf = model_e.temporal_context + 1
print(f"learned frontend: Conv encoder (1->{ENC_CHANNELS}, stride {HOP_SIZE}) "
      f"-> dilated temporal {list(TEMPORAL_DILATIONS)} -> proj({ENC_CHANNELS}->{FRONTEND_CHANNELS})")
print(f"temporal RF: {_rf} frames (~{_rf * HOP_SIZE / SAMPLE_RATE:.2f} s causal context)")

: 

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{RUN_DIR}/tb"

: 